# Enterprise Loan Risk Intelligence Platform
## Phase 4 - Machine Learning Model Development

1. Import Libraries
2. Load Processed Data
3. Train-Test Split (or load existing split)
4. Baseline Models
5. Model Evaluation
6. Cross Validation
7. Hyperparameter Tuning
8. Feature Importance
9. Final Model Selection
10. Save Best Model
11. Conclusions

In [27]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import(
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.pipeline import Pipeline

import joblib


In [29]:
df = pd.read_csv("../data/processed/loan_data_processed.csv")

In [30]:
df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,loan_status,pymnt_plan,purpose,addr_state,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,...,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,credit_history_length,loan_to_income_ratio,installment_to_income_ratio,emp_length_numeric
0,2500,2500,2500.0,36,13.56,84.92,3,11,RENT,55000.0,Not Verified,Current,0,debt_consolidation,NY,18.24,0.0,1.0,-1.0,45.0,9.0,1.0,4341,10.3,34.0,1,0.0,-1.0,1,0,0.0,0.0,16901.0,2.0,2.0,1.0,2.0,2.0,12560.0,69.0,...,34360.0,5.9,0.0,0.0,140.0,212.0,1.0,1.0,0.0,1.0,-1.0,2.0,-1.0,0.0,2.0,5.0,3.0,3.0,16.0,7.0,18.0,5.0,9.0,0.0,0.0,0.0,3.0,100.0,0.0,1.0,0.0,60124.0,16901.0,36500.0,18124.0,0,25.0,0.045455,0.001544,10
1,30000,30000,30000.0,60,18.94,777.23,4,17,MORTGAGE,90000.0,Source Verified,Current,0,debt_consolidation,LA,26.52,0.0,0.0,71.0,75.0,13.0,1.0,12315,24.2,44.0,1,0.0,-1.0,1,0,0.0,1208.0,321915.0,4.0,4.0,2.0,3.0,3.0,87153.0,88.0,...,13761.0,8.3,0.0,0.0,163.0,378.0,4.0,3.0,3.0,4.0,-1.0,4.0,-1.0,0.0,2.0,4.0,4.0,9.0,27.0,8.0,14.0,4.0,13.0,0.0,0.0,0.0,6.0,95.0,0.0,1.0,0.0,372872.0,99468.0,15000.0,94072.0,0,39.0,0.333333,0.008636,10
2,5000,5000,5000.0,36,17.97,180.69,4,16,MORTGAGE,59280.0,Source Verified,Current,0,debt_consolidation,MI,10.51,0.0,0.0,-1.0,-1.0,8.0,0.0,4599,19.1,13.0,1,0.0,-1.0,1,0,0.0,0.0,110299.0,0.0,1.0,0.0,2.0,14.0,7150.0,72.0,...,13800.0,0.0,0.0,0.0,87.0,92.0,15.0,14.0,2.0,77.0,-1.0,14.0,-1.0,0.0,0.0,3.0,3.0,3.0,4.0,6.0,7.0,3.0,8.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,136927.0,11749.0,13800.0,10000.0,0,15.0,0.084345,0.003048,6
3,4000,4000,4000.0,36,18.94,146.51,4,17,MORTGAGE,92000.0,Source Verified,Current,0,debt_consolidation,WA,16.74,0.0,0.0,-1.0,-1.0,10.0,0.0,5468,78.1,13.0,1,0.0,-1.0,1,0,0.0,686.0,305049.0,1.0,5.0,3.0,5.0,5.0,30683.0,68.0,...,1239.0,75.2,0.0,0.0,62.0,154.0,64.0,5.0,3.0,64.0,-1.0,5.0,-1.0,0.0,1.0,2.0,1.0,2.0,7.0,2.0,3.0,2.0,10.0,0.0,0.0,0.0,3.0,100.0,100.0,0.0,0.0,385183.0,36151.0,5000.0,44984.0,0,20.0,0.043478,0.001592,10
4,30000,30000,30000.0,60,16.14,731.78,3,14,MORTGAGE,57250.0,Not Verified,Current,0,debt_consolidation,MD,26.35,0.0,0.0,-1.0,-1.0,12.0,0.0,829,3.6,26.0,1,0.0,-1.0,1,0,0.0,0.0,116007.0,3.0,5.0,3.0,5.0,4.0,28845.0,89.0,...,8471.0,8.9,0.0,0.0,53.0,216.0,2.0,2.0,2.0,2.0,-1.0,13.0,-1.0,0.0,2.0,2.0,3.0,8.0,9.0,6.0,15.0,2.0,12.0,0.0,0.0,0.0,5.0,92.3,0.0,0.0,0.0,157548.0,29674.0,9300.0,32332.0,0,26.0,0.524017,0.012782,10


In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 90 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   loan_amnt                       int64  
 1   funded_amnt                     int64  
 2   funded_amnt_inv                 float64
 3   term                            int64  
 4   int_rate                        float64
 5   installment                     float64
 6   grade                           int64  
 7   sub_grade                       int64  
 8   home_ownership                  str    
 9   annual_inc                      float64
 10  verification_status             str    
 11  loan_status                     str    
 12  pymnt_plan                      int64  
 13  purpose                         str    
 14  addr_state                      str    
 15  dti                             float64
 16  delinq_2yrs                     float64
 17  inq_last_6mths                  float6